In [1]:
# What are diffusion models.?
# In diffusion models, we start with a clean image.
# Slowly add noise in it until the data becomes pure static
# Train a model to reverse this process

# So the model learns: Given noise -> reconstruct original image

In [2]:
# Forward Process (Noise scheduling)
# Forward process is also known as noise scheduling
# We gradually corrupt data using gaussian noise
# Step by step:
# At each timestep t:
# xt = sqrt(At).x(t-1) + sqrt(1-At). E
# x0: Original image, xt = Noisy image, E = random noise, At = Noise schedule

In [3]:
# Noise scheduling
# Defines how fast noise is added
# Common noise schedules are:
# - Linear (simple)
# - Cosine (better quality)
# - Quadritic (complex)

In [4]:
# Imported Pytorch
import torch

# Pre defined T or timestamp value which is equal to 1000
T = 5

# Line 1: Creates T values between 0.0001 and 0.02
# T = 5
# beta = [0.0001,0.005,0.01,0.015,0.02]
# beta = noise value at each timestamp
# small -> less noise
# large -> more noise
# Beta controls how much noise we add at each step
beta = torch.linspace(0.0001,0.02,T)

# Line 2: alpha = 1 - beta
# Subtracts each value from 1
# alpha = [0.9999,0.995,0.99,0.985,0.98]
# alpha = how much orginal value we are keeping
# if beta = noise, alpha = signal
# signal + noise = 1
# Alpha will tell you how much original image we need to preserve
alpha = 1 - beta

# Line 3: alpha_hat = torch.cumprod(alpha,dim=0)
# Takes cumulative product
# alpha_hat[0] = alpha[0], alpha_hat[1] = alpha[0]*alpha[1],alpha_hat[2] = alpha[0]*alpha[1]*alpha[2]
# alpha_hat[t] = total signal left after t steps
# It keeps multiplying -> signal keeps decreasing and noise keeps increasing
# More steps -> less original image -> more noise
#
alpha_hat = torch.cumprod(alpha,dim=0)

In [5]:
# Reverse Process (Learning to denoise)
# Model learns p0(xt-1|xt)
# Instead of predicting image directly, we predict the noise
# E0(xt,t)

# Loss function
# L = E[||E - E0(xt,t)||^2]
# True noise vs predicted noise
# Minimize difference

In [6]:
# Model Architecture (Core of diffusion)
# U-Net(Backbone)
# Why do we use U-Net.?
# Captures local + global features
# Skip connection preserve details
# Components
# - Downsampling (encoder)
# - Bottleneck
# - Upsampling (decoder)
# - Skip connections

# Time embeddings
# We inject timstep into:
# t_embedding = sinusoidal_embedding(t)

# Full pipeline
# Noisy image + timestep -> U-Net -> Output: Predicted noise

In [7]:
# Stable diffusion
# This is advanced architecture
# Stable diffusion works in latent space (not pixel space)

# Key components
# 1. VAE (Encoder + Decoder)
# - Compress images -> latent space
# - Decode back to the image

# 2. U-Net(Diffusion model)
# Works on latent representation

# 3. Text encoder(CLIP)
# Converts prompt -> embedding

# Flow:
# Text -> CLIP -> embedding
# Image -> VAE -> latent

# Latent + Noise -> U-Net -> Denoised

# Latent -> VAE decoder -> Final image


In [8]:
# Training a diffusion model
# Step 1: Install
!pip install torch torchvision matplotlib

In [9]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5,))
])

dataset = datasets.MNIST(root="./data",train=True,transform = transform,download=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 37.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.01MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.56MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.77MB/s]


In [16]:
# Step 3: Simple U-Net model
# importing pytorch neural network module
import torch.nn as nn

# Created a custom neural network class
class SimpleUNet(nn.Module):
  def __init__(self):
    super().__init__()
    print("SimpleUNet __init__ called") # Diagnostic: Confirm __init__ is run

    # Define model
    # Create a pipeline of layers
    self.model = nn.Sequential(
        nn.Conv2d(1,64,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(64,1,3,padding=1)
    )
  def forward(self,x,t):
    print(f"SimpleUNet forward called with x shape: {x.shape}") # Diagnostic: Confirm forward is run
    # The input 't' is not used in this simple U-Net, but keeping it for signature consistency
    output = self.model(x)
    print(f"Output type from self.model(x) in forward: {type(output)}") # Diagnostic: Check immediate output type
    return output

In [ ]:
# Step 4: Training loop
import torch
from torchvision import datasets, transforms

# Re-define T and alpha_hat for the training loop to ensure consistency
T = 5
beta = torch.linspace(0.0001, 0.02, T)
alpha = 1 - beta
alpha_hat = torch.cumprod(alpha, dim=0)

# Re-define transform and dataset to ensure they are correctly initialized
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5,))
])
dataset = datasets.MNIST(root="./data",train=True,transform = transform,download=True)

model = SimpleUNet()
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

for epoch in range(5):
  for x, _ in dataset:
    # Ensure x is 4D (batch_size, channels, height, width)
    # The dataset typically returns (C, H, W) for single images,
    # so unsqueeze(0) makes it (1, C, H, W).
    x = x.unsqueeze(0)

    # Sample a random timestep t for the current image
    # t should be within the range [0, T-1]
    t = torch.randint(0, T, (1,)).long() # .long() is important for indexing

    # Generate random noise
    noise = torch.randn_like(x)

    # Calculate the noisy image using the forward diffusion process formula:
    # noisy_x = sqrt(alpha_hat[t]) * x_0 + sqrt(1 - alpha_hat[t]) * noise
    sqrt_alpha_hat_t = torch.sqrt(alpha_hat[t]).view(-1, 1, 1, 1)
    sqrt_one_minus_alpha_hat_t = torch.sqrt(1 - alpha_hat[t]).view(-1, 1, 1, 1)

    noisy_x = sqrt_alpha_hat_t * x + sqrt_one_minus_alpha_hat_t * noise

    # Predict the noise using the U-Net model
    pred_noise = model(noisy_x, t)

    # Diagnostic: Add a print statement to verify the type of pred_noise right before the error
    print(f"Type of pred_noise before loss calculation: {type(pred_noise)}")

    # Calculate the loss (mean squared error between true noise and predicted noise)
    loss = ((noise - pred_noise) ** 2).mean()

    # Backpropagation and optimization step
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f"Epoch {epoch}, Loss : {loss.item()}")

# Step 5: Sampling

Streaming output truncated to the last 5000 lines.
Type of pred_noise before loss calculation: <class 'torch.Tensor'>
SimpleUNet forward called with x shape: torch.Size([1, 1, 28, 28])
Output type from self.model(x) in forward: <class 'torch.Tensor'>
Type of pred_noise before loss calculation: <class 'torch.Tensor'>
SimpleUNet forward called with x shape: torch.Size([1, 1, 28, 28])
Output type from self.model(x) in forward: <class 'torch.Tensor'>
Type of pred_noise before loss calculation: <class 'torch.Tensor'>
SimpleUNet forward called with x shape: torch.Size([1, 1, 28, 28])
Output type from self.model(x) in forward: <class 'torch.Tensor'>
Type of pred_noise before loss calculation: <class 'torch.Tensor'>
SimpleUNet forward called with x shape: torch.Size([1, 1, 28, 28])
Output type from self.model(x) in forward: <class 'torch.Tensor'>
Type of pred_noise before loss calculation: <class 'torch.Tensor'>
SimpleUNet forward called with x shape: torch.Size([1, 1, 28, 28])
Output type fro